# Treino — BTC-USD (Yahoo Finance)

Baixa o histórico diário de Bitcoin (**05/10/2021** a **05/10/2026**), treina uma **regressão linear** (equivalente a `LinearRegression`) para estimar o **close de amanhã** e exporta `models/btc_model.joblib`.

Mesmo princípio do exemplo com ações:

```python
import yfinance as yf
data = yf.download("BTC-USD", start="2021-10-05", end="2026-10-06")
data.to_csv("data/btc_usd_dados.csv")
```

> No `yfinance`, `end` é exclusivo — por isso `2026-10-06` inclui o dia 05/10/2026.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yfinance as yf

## 1. Baixar dados do Yahoo Finance e salvar CSV

In [2]:
ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

data = yf.download("BTC-USD", start="2021-10-05", end="2026-10-06", auto_adjust=True)
csv_path = DATA_DIR / "btc_usd_dados.csv"
data.to_csv(csv_path)
print(f"Linhas baixadas: {len(data)}")
print(f"CSV salvo em: {csv_path}")
data.tail()

[*********************100%***********************]  1 of 1 completed

Linhas baixadas: 1827
CSV salvo em: C:\Users\ccl19\OneDrive\Documentos\projetos\Ponderada05-10\data\btc_usd_dados.csv


Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,
2026-10-01,84853.101562,85224.273438,83132.757812,83554.250000,32802609098
2026-10-02,84497.210938,87146.351562,83852.695312,84855.648438,46528418756
2026-10-03,84763.578125,85019.328125,84429.648438,84503.187500,12574826191
2026-10-04,86480.304688,86766.148438,84707.320312,84765.515625,15031650242
2026-10-05,85461.359375,86929.656250,85098.796875,86513.382812,31068450816


## 2. Features e alvo (close de amanhã)

In [3]:
frame = data.copy()
if isinstance(frame.columns, pd.MultiIndex):
    frame.columns = [c[0] for c in frame.columns]
frame = frame.reset_index()
date_col = "Date" if "Date" in frame.columns else frame.columns[0]

df = pd.DataFrame({
    "Date": pd.to_datetime(frame[date_col], errors="coerce"),
    "Close": pd.to_numeric(frame["Close"], errors="coerce"),
}).dropna().sort_values("Date").reset_index(drop=True)

df["close"] = df["Close"]
df["return_1d"] = df["close"].pct_change()
df["ma_7"] = df["close"].rolling(7).mean()
df["target_next_close"] = df["close"].shift(-1)

feature_cols = ["close", "return_1d", "ma_7"]
model_df = df.dropna(subset=feature_cols + ["target_next_close"]).copy()
print(f"Amostras após features: {len(model_df)}")
print(f"Período features: {model_df['Date'].iloc[0].date()} -> {model_df['Date'].iloc[-1].date()}")
model_df[["Date"] + feature_cols + ["target_next_close"]].tail()

Amostras após features: 1820
Período features: 2021-10-11 -> 2026-10-04


,Date,close,return_1d,ma_7,target_next_close
1821,2026-09-30,83553.851562,-0.000820,83993.916295,84853.101562
1822,2026-10-01,84853.101562,0.015550,84061.636161,84497.210938
1823,2026-10-02,84497.210938,-0.004194,84127.677455,84763.578125
1824,2026-10-03,84763.578125,0.003152,84178.695312,86480.304688
1825,2026-10-04,86480.304688,0.020253,84467.583705,85461.359375


## 3. Split cronológico + regressão linear

In [4]:
split_idx = int(len(model_df) * 0.8)
train = model_df.iloc[:split_idx]
test = model_df.iloc[split_idx:]

X_train = train[feature_cols].to_numpy(dtype=float)
y_train = train["target_next_close"].to_numpy(dtype=float)
X_test = test[feature_cols].to_numpy(dtype=float)
y_test = test["target_next_close"].to_numpy(dtype=float)

# Equivalente a sklearn.linear_model.LinearRegression().fit(...)
ones = np.ones((X_train.shape[0], 1))
beta, *_ = np.linalg.lstsq(np.hstack([ones, X_train]), y_train, rcond=None)
intercept = float(beta[0])
coef = beta[1:]

preds = X_test @ coef + intercept
mae = float(np.mean(np.abs(y_test - preds)))
ss_res = float(np.sum((y_test - preds) ** 2))
ss_tot = float(np.sum((y_test - np.mean(y_test)) ** 2))
r2 = 1.0 - ss_res / ss_tot if ss_tot else float("nan")

print(f"Treino: {len(train)} | Teste: {len(test)}")
print(f"MAE (teste): {mae:.2f}")
print(f"R2 (teste): {r2:.4f}")
print("Coeficientes:", dict(zip(feature_cols, coef)))
print("Intercepto:", intercept)

Treino: 1456 | Teste: 364
MAE (teste): 1308.46
R2 (teste): 0.9826
Coeficientes: {'close': np.float64(0.9806948288229312), 'return_1d': np.float64(-1542.3550245152849), 'ma_7': np.float64(0.020177192985199155)}
Intercepto: 3.6566079597327303


## 4. Exportar artefato

In [5]:
artifact = {
    "coef": coef.tolist(),
    "intercept": intercept,
    "feature_cols": feature_cols,
    "ticker": "BTC-USD",
    "horizon": "next_day_close",
    "model_type": "LinearRegression",
}
model_path = MODELS_DIR / "btc_model.joblib"
joblib.dump(artifact, model_path)
print(f"Modelo salvo em: {model_path}")

Modelo salvo em: C:\Users\ccl19\OneDrive\Documentos\projetos\Ponderada05-10\models\btc_model.joblib
